# FRM · Phase 1 — Teacher Label Generation (GPU, one-time, expensive)

Runs the FROZEN SmolVLM2-2.2B over every gazed WearVQA sample to cache the global tokens `G[64,d]` and the attention-rollout importance labels `imp_answer / imp_question / imp_context`. Resumable — safe to re-run. **Requires a GPU runtime.** Do this before any experiment.

In [1]:
# 1) mount Google Drive (dataset + outputs live here)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2) clone the repo containing frm/  (EDIT REPO_URL if your repo name differs)
import os
REPO_URL = "https://github.com/shubhamOjha1000/AAAI_2027_code.git"
REPO_DIR = "/content/AAAI_2027_code"
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
# locate frm/ (it sits at the repo root)
FRM = os.path.join(REPO_DIR, "frm")
if not os.path.exists(os.path.join(FRM, 'config.py')):
    import subprocess
    hit = subprocess.check_output(['bash','-lc',
        f"find {REPO_DIR} -name config.py -path '*frm*' | head -1"]).decode().strip()
    FRM = os.path.dirname(hit)
assert os.path.exists(os.path.join(FRM, "config.py")), "frm/ not found — check REPO_URL"
%cd $FRM

Cloning into '/content/AAAI_2027_code'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 228 (delta 99), reused 200 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (228/228), 308.09 KiB | 4.60 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/AAAI_2027_code/frm


In [3]:
# 3) install deps
!pip -q install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 5.8 MB/s eta 0:00:00


In [4]:
# 4) point at the dataset on Drive + output dir; put frm/ on the path
import os, sys
os.environ["DATA_DIR"] = "/content/drive/MyDrive/wearvqa_gaze_only"
os.environ["FRM_OUT_DIR"] = "/content/drive/MyDrive/frm_out"
sys.path.insert(0, os.getcwd())
import config as C; print('DATA_DIR =', C.DATA_DIR); print('OUT_DIR  =', C.OUT_DIR)

DATA_DIR = /content/drive/MyDrive/wearvqa_gaze_only
OUT_DIR  = /content/drive/MyDrive/frm_out


### GPU check

In [5]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-10c0d377-9ebc-f562-6928-7c9545777e83)


### Smoke test (5 samples) — verify the VLM + rollout path before the full run

In [6]:
import labels
labels.generate(limit=5)

total=5 done=0 todo=5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

label-gen:   0%|          | 0/5 [00:00<?, ?it/s]


AssertionError: got 81 image tokens, expected 64

### Full run (all ~1000 samples). Resumes from where it stopped.

In [ ]:
import importlib, labels; importlib.reload(labels)
labels.generate()

### Peek at cached labels

In [ ]:
from labels import load_labels
metas, G, imp = load_labels()
print('samples:', len(metas), '| G shape:', G.shape)
print('imp_answer[0][:8]:', imp['imp_answer'][0][:8])